<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 8.2 — RAG Foundations: Embeddings + Retrieval + LLM (EPI 264)

In this notebook, you will build a simple Retrieval-Augmented Generation (RAG) workflow over de-identified clinical notes prepared in Lab 6.

### What you will do
1) Load cleaned notes (Lab 6 output)
2) Embed notes using MiniLM (local embeddings)
3) Use FAISS to retrieve the most relevant notes for a query
4) Provide retrieved notes to a local LLM (Ollama) to extract a **dementia phenotype (Yes/No)**

### Why this matters
Dementia is often under-coded in structured diagnosis tables. RAG lets us search narrative notes for phenotyping signals that structured data may miss.

## 1. Prepare Notes Data for RAG

We will load the cleaned notes dataset from Lab 6 (deduplicated and restricted to the cohort window).

**Important:**
We will NOT use physician gold labels inside the retrieval or prompt context (to avoid leakage).
Gold labels are only used later for evaluation.

In [1]:
# -----------------------------------------------------------
# 1.1. Load Cleaned Notes + Evaluation Labels (Lab 6 Outputs)
# -----------------------------------------------------------
# Inputs:
# - lab6_clean_notes_baseline.parquet  (note-level dataset for RAG)
# - lab6_structured_dementia_eval.csv  (patient-level labels for evaluation only)
#
# Goal:
# - notes_df: used for embeddings + retrieval (NO gold labels attached)
# - eval_df: used later to evaluate RAG outputs
# -----------------------------------------------------------

from IPython.display import display, Markdown
from pathlib import Path
import pandas as pd

filepath = Path("data_files")

notes_df = pd.read_parquet(filepath / "lab6_clean_notes_baseline.parquet")
eval_df  = pd.read_csv(filepath / "lab6_structured_dementia_eval.csv")

print("Notes shape:", notes_df.shape)
print("Eval shape :", eval_df.shape)

display(notes_df.head(10))

Notes shape: (2639, 7)
Eval shape : (97, 6)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,The patient continues to experience shoulder p...
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,Sylvia was seen for follow-up regarding her sy...
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,The patient is a 69-year-old woman with a medi...
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,The patient presents with a large hiatal herni...
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This progress note documents ongoing cardiolog...
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,The care manager reached out to the patient an...
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient presented for a blood pressure che...
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\n\nThe patient is a 75-year-...
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presents for a one-year follow-up ...


## 2. Embed and Store Clinical Notes in a FAISS Vector Store (In-Memory)

In this step, we embed full clinical notes and store them in a **FAISS** (Facebook AI Similarity Search
) vector store, which enables efficient similarity search. We use a lightweight transformer model (`MiniLM`) to convert each note into a semantic vector.

### Steps:
- **2.1**: Embed clinical notes using the `all-MiniLM-L6-v2` model from Hugging Face.
- **2.2**: View an embedded document along with its metadata and vector representation.

### Why Use This Approach?

Storing entire notes is useful when:
- You want to preserve the full clinical context for each patient.
- Your downstream use case (e.g., summarization or structured extraction) requires complete narrative input.
- The notes are concise enough to fit within the input limits of an LLM.

This method simplifies retrieval workflows by allowing you to work with whole documents rather than fragmented chunks.

<img src="./images/rag_full.png" alt="RAG Full" width="900">




In [2]:
# -----------------------------------------------------------
# 2.1. Embed Clinical Notes Using Local MiniLM Embeddings
# -----------------------------------------------------------
# This cell encodes each clinical note into a vector using a local
# transformer model and stores those embeddings in a FAISS index for
# fast similarity search.

# Model: `sentence-transformers/all-MiniLM-L6-v2`
# - Optimized for semantic similarity tasks
# - Lightweight and fast (384-dimensional vectors)
# - Runs fully offline
# -----------------------------------------------------------

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Prepare inputs for embedding: note text and relevant metadata
documents = notes_df["note_txt_deid"].tolist()
metadata = notes_df[["patient_num", "visit_id", "create_dts_shifted", "note_csn_id" ,"inpatient_note_type_dsc"]].to_dict(orient="records")

# Create a FAISS vector store from the documents
vectorstore = FAISS.from_texts(
    documents,
    embedding_model,
    metadatas=metadata
)

print(f"✅ Successfully embedded {len(documents)} clinical notes using MiniLM.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Successfully embedded 2639 clinical notes using MiniLM.


In [3]:
# -----------------------------------------------------------
# 2.2. View a Specific Embedded Document, Metadata, and Vector
# -----------------------------------------------------------
# Select an index (e.g., id = 5) to inspect the stored document.
# This cell shows the document text, associated metadata, and
# the corresponding FAISS embedding vector.
# -----------------------------------------------------------

id = 1  # You can change this index to view a different record

# Get (doc_id, Document) tuple from LangChain's docstore
doc_id, doc_example = list(vectorstore.docstore._dict.items())[id]

# Retrieve corresponding vector from FAISS
vector_example = vectorstore.index.reconstruct(id)

display(Markdown(f"### 🧾 Document ID: `{doc_id}`"))
display(Markdown(f"**Metadata:** `{doc_example.metadata}`"))

display(Markdown("**Document Text (First 500 characters):**"))
display(Markdown(f"```text\n{doc_example.page_content}...\n```"))

display(Markdown("**Embedded Vector (First 100):**"))
display(Markdown(f"```text\n{vector_example[:100]}\n```"))

### 🧾 Document ID: `a8ab2a05-4d1e-4eff-bc1f-d871f71acb3a`

**Metadata:** `{'patient_num': 'H111336931', 'visit_id': 6332144391, 'create_dts_shifted': Timestamp('2017-01-01 17:11:00'), 'note_csn_id': 2712892487, 'inpatient_note_type_dsc': 'Progress Notes'}`

**Document Text (First 500 characters):**

```text
Sylvia was seen for follow-up regarding her systemic lupus erythematosus (SLE), after not having been evaluated for approximately 18 months. She recently had an appointment with another provider and was found to have new skin lesions developing on her hands, which are asymptomatic but have progressed over time. She consulted with a dermatologist at the time these lesions appeared and was prescribed topical steroids, which provided minimal benefit. She notes that changes in the color of the lesions seem exacerbated by cold, although she does not report classic symptoms of Raynaud's phenomenon.

Review of systems reveals a history of chronic kidney disease, stable, SLE with cutaneous features, and intermittent mild thrombocytopenia. There is no evidence of other organ involvement, and she continues to be physically active. Current medications include ProAir inhaler and hydroxychloroquine 200 mg daily.

On examination, her blood pressure was 147/67, pulse 63, and temperature 97. Head and neck exam was unremarkable, and the oral cavity was normal. A mild malar flush was observed. Joint evaluation did not find any active synovitis; shoulders, elbows, wrists, hands, hips, knees, ankles, and feet appeared healthy. Closer inspection of her hands revealed macular and papular eruptions, some of which blanch and are located periungually around the nailbeds. These lesions resemble chilblain in appearance.

Clinical impression is SLE with primarily cutaneous involvement and new skin lesions that are likely chilblain, which often resolve with warmer temperatures and are more common in winter. Given that the lesions are not bothersome, it is recommended to monitor her symptoms through the winter, and reassess in the spring if there is no improvement. Hydroxychloroquine was continued as it may be beneficial for her condition, and more potent immunosuppressive therapies such as CellCept or azathioprine were considered unnecessary. Routine follow-up is planned in one year unless the lesions persist or worsen....
```

**Embedded Vector (First 100):**

```text
[-0.06820683 -0.01494661  0.01385453  0.03297242 -0.04186773 -0.05952919
  0.03221074  0.15806589 -0.01082682 -0.01893152 -0.02991626  0.04575785
  0.05017426  0.02686733 -0.02597007  0.01745564  0.03755568 -0.07546723
  0.02510483  0.0876174  -0.03540619 -0.00057738 -0.06145635 -0.03833029
  0.02967867  0.06094506 -0.03124153 -0.03629926 -0.0194319   0.05715854
 -0.02968677  0.04641811 -0.13212475  0.00741137 -0.00406911  0.02037557
  0.00237211  0.03395102 -0.06550582  0.06356963  0.0142055  -0.09916136
 -0.05174129 -0.03501055 -0.05425777 -0.02260536 -0.03371664  0.12057907
  0.01870984 -0.02323652  0.01430013 -0.06046596  0.0239399  -0.01826104
 -0.03736203 -0.00147769 -0.02547079 -0.08179823 -0.00144504 -0.0378483
 -0.06299665  0.00563113  0.01587242 -0.01841112  0.00840955 -0.05165416
  0.05896427 -0.07977705  0.07497375  0.05837149  0.04090753  0.02381499
  0.01947917  0.03985326  0.01440708  0.02644934  0.03094163 -0.02992491
 -0.02509333 -0.01944447  0.07194928  0.08077575  0.0456391   0.02204424
  0.03440748  0.04621087 -0.02504835 -0.01617088 -0.08857316 -0.00946128
 -0.01278373  0.00401419 -0.01379466 -0.05144616  0.02347013  0.01923481
 -0.02744811 -0.00419847 -0.05962082  0.00604658]
```

## 3. Retrieving Clinical Notes with Similarity Score (RAG Retrieval)

In this section, we perform semantic search over embedded clinical notes using a FAISS vector store and a locally generated query vector. We use similarity scores to evaluate the relevance of each match to the query.

### Key Retrieval Steps:

1. **Embed a Query (Step 3.1)**
   - Converts a natural language question into a numerical vector using the same MiniLM model used to embed the notes.

2. **Similarity Search with Scores (Step 3.2)**
   - Retrieves the top-k clinical notes ranked by cosine similarity to the query.
   - Includes similarity scores for transparency and ranking.

3. **Score Threshold Filtering (Step 3.3)**
   - Filters out matches with low similarity scores.
   - Helps improve the precision and clinical relevance of the results.

### Why Use These Techniques?

Similarity search helps identify notes most relevant to a user-defined question or condition. Threshold filtering ensures:
- Only strong matches are considered for downstream tasks like summarization
- Noisy or unrelated content is excluded
- Each result can be justified based on a similarity score

<img src="./images/rag_retrieval.png" alt="RAG Retrieval" width="900">


In [4]:
# -----------------------------------------------------------
# 3.1. Embed a Query and Inspect Its Vector Representation
# -----------------------------------------------------------
# This step encodes a natural language query into a numerical vector
# using the same MiniLM model used for the clinical notes.
# This vector will be used to search for semantically similar notes.
# -----------------------------------------------------------

# Define a sample clinical query
query = "Which patients have documented dementia or Alzheimer disease?"

# Generate the embedding for the query
query_vector = embedding_model.embed_query(query)

# Display the vector and its shape
display(Markdown("### Vectorized Query"))
display(Markdown(f"`Query:` *{query}*"))
display(Markdown("**Embedding Vector (truncated):**"))
display(Markdown(f"```text\n{query_vector[:100]} ... [{len(query_vector)} dimensions]\n```"))


### Vectorized Query

`Query:` *Which patients have documented dementia or Alzheimer disease?*

**Embedding Vector (truncated):**

```text
[0.011906935833394527, 0.03944014757871628, -0.09752410650253296, 0.06022509187459946, -0.04667927324771881, 0.04867030680179596, -0.05022354796528816, 0.029064081609249115, -0.028790153563022614, -0.023942913860082626, 0.03695804253220558, 0.05865553021430969, -0.02685238979756832, 0.049717485904693604, -0.051887545734643936, 0.05872119963169098, -0.08355791121721268, 0.04715631902217865, 0.06059790030121803, 0.02870223857462406, -0.020331967622041702, 0.023083791136741638, 0.06421563029289246, -0.0038007257971912622, 0.02896592766046524, -0.021871929988265038, -0.05860956758260727, -0.086844801902771, 0.012490646913647652, 0.0435485802590847, 0.005426767282187939, 0.04009679704904556, -0.049031179398298264, 0.040420711040496826, -0.0029337992891669273, -0.042801544070243835, -0.037239041179418564, 0.07821102440357208, -0.045647356659173965, -0.03393026441335678, -0.012661042623221874, 0.016520781442523003, 0.07573212683200836, -0.0023874626494944096, -0.01474627386778593, -0.04000601917505264, -0.01247819047421217, -0.010556485503911972, -0.01922827586531639, -0.004239893052726984, -0.0685100182890892, -0.03398379310965538, 0.005000611301511526, 0.04402044042944908, 0.033297717571258545, 0.019468585029244423, 0.03023684397339821, -0.03426550328731537, -0.08062276989221573, 0.03611023724079132, -0.06497181951999664, -0.04285372421145439, -0.03309416025876999, 0.007066899444907904, -0.038591545075178146, 0.0902220755815506, -0.028005165979266167, -0.05437342822551727, -0.0321253165602684, -0.07484523206949234, -0.02471080794930458, -0.0031285732984542847, 0.029322747141122818, 0.06368803977966309, 0.013924732804298401, -0.01169080100953579, -0.035682786256074905, 0.0597379207611084, -0.01746251806616783, -0.13445661962032318, 0.0181600209325552, 0.03753931447863579, 0.04464970901608467, -0.004935245029628277, 0.11035894602537155, 4.173434717813507e-05, 0.0027889148332178593, 0.03659074753522873, -0.07208330184221268, -0.013018954545259476, 0.04452178627252579, -0.00509884487837553, -0.06256204098463058, -0.06569273769855499, -0.05408468097448349, -0.011294152587652206, 0.04169526323676109, 0.03355439007282257, -0.02642577514052391, 0.002572137862443924] ... [384 dimensions]
```

In [5]:
# -----------------------------------------------------------
# 3.2. Similarity Search (Top-K Results, No Filtering)
# -----------------------------------------------------------
# This cell performs a semantic similarity search using the embedded query,
# returning the top-k most similar clinical notes along with similarity scores.

# Score Interpretation (Euclidean Distance):
# - 0.00 – 0.30: Highly relevant
# - 0.30 – 0.50: Strong match
# - 0.50 – 0.70: Moderate match
# - 0.70 – 0.90: Low match
# - 0.90+: Minimal or irrelevant

# similarity_score = 1 / (1 + distance)  # Rescales Euclidean Distance into similarity-like score
# -----------------------------------------------------------

# Define number of top results
top_k = 10

# Run similarity search
results = vectorstore.similarity_search_with_score(query, k=top_k)

# Display header
display(Markdown(f"### 🔍 Top {top_k} Most Similar Clinical Notes"))

# Iterate and display each match
for i, (doc, score) in enumerate(results):
    display(Markdown(f"---\n**Result {i+1}**  \n- **Distance Score:** `{score:.4f}`  \n- **Patient Num:** `{doc.metadata.get('patient_num', 'N/A')}`  \n- **Visit ID:** `{doc.metadata.get('visit_id', 'N/A')}`\n\n**Note Preview:**\n```text\n{doc.page_content[:1200]}\n```"))


### 🔍 Top 10 Most Similar Clinical Notes

---
**Result 1**  
- **Distance Score:** `0.6207`  
- **Patient Num:** `H116882795`  
- **Visit ID:** `6371645545`

**Note Preview:**
```text
The provider has reviewed the recent notes, assessments, and procedures documented by the nurse practitioner and concurs with their clinical findings. The patient's prior diagnosis of dementia was acknowledged in the reviewed documentation.
```

---
**Result 2**  
- **Distance Score:** `0.8362`  
- **Patient Num:** `H116882795`  
- **Visit ID:** `6466937973`

**Note Preview:**
```text
The patient is stable and responding favorably to the present treatment plan. Previous medical history includes a highly established diagnosis of dementia. No new concerns noted at this time. Continue current management.
```

---
**Result 3**  
- **Distance Score:** `0.8619`  
- **Patient Num:** `H115124574`  
- **Visit ID:** `6369691025`

**Note Preview:**
```text
The patient was admitted to the inpatient service on 8/24/2017. Communication with the inpatient care team was established by phone call, as preferred, and the case manager was contacted. The initial concern included anemia and consideration of heart failure. Based on previously documented information, a prior history or working diagnosis of dementia is noted. The patient is able to be contacted directly.
```

---
**Result 4**  
- **Distance Score:** `0.8699`  
- **Patient Num:** `H117326759`  
- **Visit ID:** `6432799125`

**Note Preview:**
```text
Medication orders were reviewed, and reconciliation was performed with the health record on 9/23/17. The dates of service were from 8/25/17 to 10/24/17. This progress note reflects ongoing monitoring, which is especially important given the patient's prior diagnosis of dementia.
```

...

---
**Result 6**  
- **Distance Score:** `0.9175`  
- **Patient Num:** `H115124574`  
- **Visit ID:** `6412979563`

**Note Preview:**
```text
A telephone call was made to the patient and spouse to arrange an initial visit. An appointment was scheduled for 3/18 at 11am, and relevant paperwork is in place. The most recent INR measurement was on 2/19, with home care services involved at the last visit. The patient is expected to visit the laboratory following the upcoming appointment. There is a prior working diagnosis of dementia that will be considered during ongoing care.
```

---
**Result 7**  
- **Distance Score:** `0.9342`  
- **Patient Num:** `H115124574`  
- **Visit ID:** `6362274807`

**Note Preview:**
```text
On 7/21/2017, the patient was admitted to the inpatient unit. Lip swelling was noted at the time of admission. The care management team was able to contact the patient, and communication was established. The assessment reflects mild confidence that the patient carries a prior diagnosis of dementia, though further clarification is needed. No additional interval events or findings are documented in this note.
```

---
**Result 8**  
- **Distance Score:** `0.9635`  
- **Patient Num:** `H116088887`  
- **Visit ID:** `6452776567`

**Note Preview:**
```text
Received a call from dental staff regarding an upcoming dental evaluation for concern of a possible infected tooth. An updated medication and allergy list was provided to the dental provider as requested. There is a known prior diagnosis of dementia. Awaiting further information from the dental team following their assessment.
```

---
**Result 9**  
- **Distance Score:** `0.9657`  
- **Patient Num:** `H117408294`  
- **Visit ID:** `6332324331`

**Note Preview:**
```text
The patient currently does not wish to pursue interventions. This decision is respected and care will continue to be guided by patient preferences. A history consistent with dementia remains part of the ongoing assessment.
```

---
**Result 10**  
- **Distance Score:** `0.9660`  
- **Patient Num:** `H115124574`  
- **Visit ID:** `6381585861`

**Note Preview:**
```text
The patient was transferred to the emergency department on 10/17/17 due to congestive heart failure and was subsequently admitted to the inpatient unit. There is a background of cognitive concerns, with a prior working diagnosis of dementia considered with mild confidence.
```

In [6]:
# -----------------------------------------------------------
# 3.3. Filter Search Results by Similarity Score Threshold
# -----------------------------------------------------------
# This cell filters the top-K search results to keep only those
# with high similarity scores above a defined threshold.

# Similarity Score Threshold:
# - Only notes with scores ≥ threshold will be retained.
# - Higher scores = greater semantic similarity.
# -----------------------------------------------------------


from IPython.display import Markdown, display

threshold = 0.9  # Keep notes with score ≥ x

# Initialize an empty list to hold the filtered results
filtered_results = []

# Loop through each result (a tuple of Document and score)

for doc, distance in results:
    # Check if the similarity score meets the threshold
    if distance <= threshold:
        # If so, add it to the filtered list
        filtered_results.append((doc, distance))


# Summary
display(Markdown(f"### ✅ {len(filtered_results)} of {top_k} notes passed the distance threshold (<= {threshold})"))

# Show filtered results
for i, (doc, similarity) in enumerate(filtered_results):
    display(Markdown(
        f"---\n**Filtered Match {i+1}**  \n"
        f"- **Distance Score:** `{similarity:.4f}`  \n"
        f"- **Patient Num:** `{doc.metadata.get('patient_num', 'N/A')}`  \n"
        f"- **Visit ID:** `{doc.metadata.get('vist_id', 'N/A')}`\n\n"
        f"**Note Preview:**\n```text\n{doc.page_content[:1200]}\n```"
    ))


### ✅ 5 of 10 notes passed the distance threshold (<= 0.9)

---
**Filtered Match 1**  
- **Distance Score:** `0.6207`  
- **Patient Num:** `H116882795`  
- **Visit ID:** `N/A`

**Note Preview:**
```text
The provider has reviewed the recent notes, assessments, and procedures documented by the nurse practitioner and concurs with their clinical findings. The patient's prior diagnosis of dementia was acknowledged in the reviewed documentation.
```

---
**Filtered Match 2**  
- **Distance Score:** `0.8362`  
- **Patient Num:** `H116882795`  
- **Visit ID:** `N/A`

**Note Preview:**
```text
The patient is stable and responding favorably to the present treatment plan. Previous medical history includes a highly established diagnosis of dementia. No new concerns noted at this time. Continue current management.
```

---
**Filtered Match 3**  
- **Distance Score:** `0.8619`  
- **Patient Num:** `H115124574`  
- **Visit ID:** `N/A`

**Note Preview:**
```text
The patient was admitted to the inpatient service on 8/24/2017. Communication with the inpatient care team was established by phone call, as preferred, and the case manager was contacted. The initial concern included anemia and consideration of heart failure. Based on previously documented information, a prior history or working diagnosis of dementia is noted. The patient is able to be contacted directly.
```

---
**Filtered Match 4**  
- **Distance Score:** `0.8699`  
- **Patient Num:** `H117326759`  
- **Visit ID:** `N/A`

**Note Preview:**
```text
Medication orders were reviewed, and reconciliation was performed with the health record on 9/23/17. The dates of service were from 8/25/17 to 10/24/17. This progress note reflects ongoing monitoring, which is especially important given the patient's prior diagnosis of dementia.
```

---
**Filtered Match 5**  
- **Distance Score:** `0.8872`  
- **Patient Num:** `H115124574`  
- **Visit ID:** `N/A`

**Note Preview:**
```text
The patient was admitted to the inpatient unit on 12/4/2017 due to shortness of breath and congestive heart failure. Communication with the care management team was established on 12/4/2017. There is a prior diagnosis of dementia noted, for which clinical confidence is moderate. The patient remains hospitalized and is able to be contacted.
```

## 4. Generating Structured Responses with an LLM (RAG Generation)

In this section, we take the clinical notes retrieved via semantic search and pass them into a Large Language Model (LLM) to generate structured, clinically meaningful responses. This is the final step in the **Retrieval-Augmented Generation (RAG)** pipeline.

### Key Steps:

1. **Creating a Prompt Template for LLM Querying (Step 4.1)**
   - Defines a reusable prompt structure for analyzing and summarizing clinical notes.
   - Ensures each response includes patient metadata and clear, structured outputs.

2. **Invoking LLM with Retrieved Context (Step 4.2)**
   - Inserts the top-retrieved clinical notes into the prompt.
   - Sends the prompt to a local LLM (e.g., Qwen2 via Ollama) for structured generation.
   - Returns a summary that directly answers the user’s medical query.

### Why This Matters

This step demonstrates how LLMs can synthesize information from real patient notes to produce:
- Patient-specific summaries
- Answered clinical questions
- Traceable outputs with structured identifiers

This capability is essential for use cases like clinical decision support, patient-facing summaries, or intelligent search interfaces.

<img src="./images/rag_generation.png" alt="RAG Generation" width="1250">


In [7]:
# -----------------------------------------------------------
# 4.1. Prompt Template: Dementia Phenotype Extraction (Yes/No)
# -----------------------------------------------------------
# We will ask the LLM to phenotype dementia using ONLY retrieved note text.
# No gold labels are included in the prompt.
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a clinical phenotyping assistant. "
     "Use ONLY the provided notes. Do NOT infer or invent. "
     "Do NOT mention privacy/redaction/date-shifting metadata."),
    ("human",
     "Retrieved notes:\n\n{retrieved_docs}\n\n"
     "Clinical question: {query}\n\n"
     "Return a structured answer using bullet points:\n"
     "1) Dementia Phenotype (Yes/No)\n"
     "2) Evidence: up to 3 short direct quotes/phrases from the notes (or 'None')\n"
     "3) Rationale: 1–2 sentences grounded in the evidence\n"
     "4) Confidence: low/medium/high\n"
     "\nRules for Dementia Phenotype:\n"
     "- Answer 'Yes' ONLY if dementia is explicitly documented (e.g., 'dementia', 'Alzheimer', "
     "'vascular dementia', 'Lewy body dementia') or clearly stated as a prior diagnosis.\n"
     "- If dementia is not explicitly mentioned, answer 'No' (do not infer from memory complaints alone)."
    )
])

print(">> Prompt created and ready to use.")

>> Prompt created and ready to use.


In [8]:
# -----------------------------------------------------------
# 4.2. Run the Phenotype Prompt on ALL Filtered Retrieved Notes
# -----------------------------------------------------------
# We include identifiers (patient_num, visit_id, note_csn_id) for traceability.
# We pass all filtered results (after similarity threshold) into the prompt.
# -----------------------------------------------------------

from IPython.display import display, Markdown
from langchain_ollama import ChatOllama

# Safety: limit how much text per note we pass to the model
MAX_CHARS_PER_NOTE = 2500

if len(filtered_results) == 0:
    display(Markdown("### No notes passed the similarity threshold."))
else:
    # Build retrieved docs text with identifiers
    retrieved_blocks = []
    for rank, (doc, score) in enumerate(filtered_results, start=1):
        meta = doc.metadata or {}

        patient_num = meta.get("patient_num", "N/A")
        visit_id = meta.get("visit_id", "N/A")
        note_csn_id = meta.get("note_csn_id", "N/A")
        note_type = meta.get("inpatient_note_type_dsc", "N/A")
        create_dts = meta.get("create_dts_shifted", "N/A")

        note_text = (doc.page_content or "").strip()
        note_text = note_text[:MAX_CHARS_PER_NOTE]

        retrieved_blocks.append(
            f"---\n"
            f"[Retrieved Note #{rank} | distance={score:.4f}]\n"
            f"patient_num={patient_num} | visit_id={visit_id} | note_csn_id={note_csn_id}\n"
            f"note_type={note_type} | create_dts_shifted={create_dts}\n\n"
            f"{note_text}\n"
        )

    retrieved_context = "\n".join(retrieved_blocks)

    # Show what will be sent (optional but useful in workshop)
    display(Markdown(f"### Retrieved notes sent to the LLM: {len(filtered_results)}"))
    display(Markdown(retrieved_context[:3000] + ("\n\n*(preview truncated)*" if len(retrieved_context) > 3000 else "")))

    # Fill prompt
    final_prompt = prompt_template.format(
        retrieved_docs=retrieved_context,
        query=query
    )

    # Invoke local model (deterministic)
    model = ChatOllama(model="gemma3:12b", temperature=0.0)
    response = model.invoke(final_prompt)

    display(Markdown("### 📋 LLM Output: Dementia Phenotype from Retrieved Notes"))
    display(Markdown(response.content))

### Retrieved notes sent to the LLM: 5

---
[Retrieved Note #1 | distance=0.6207]
patient_num=H116882795 | visit_id=6371645545 | note_csn_id=3115119523
note_type=Progress Notes | create_dts_shifted=2017-08-27 21:08:00

The provider has reviewed the recent notes, assessments, and procedures documented by the nurse practitioner and concurs with their clinical findings. The patient's prior diagnosis of dementia was acknowledged in the reviewed documentation.

---
[Retrieved Note #2 | distance=0.8362]
patient_num=H116882795 | visit_id=6466937973 | note_csn_id=4414526153
note_type=Assessment & Plan Note | create_dts_shifted=2018-12-29 17:34:00

The patient is stable and responding favorably to the present treatment plan. Previous medical history includes a highly established diagnosis of dementia. No new concerns noted at this time. Continue current management.

---
[Retrieved Note #3 | distance=0.8619]
patient_num=H115124574 | visit_id=6369691025 | note_csn_id=3088256771
note_type=Progress Notes | create_dts_shifted=2017-08-24 12:26:00

The patient was admitted to the inpatient service on 8/24/2017. Communication with the inpatient care team was established by phone call, as preferred, and the case manager was contacted. The initial concern included anemia and consideration of heart failure. Based on previously documented information, a prior history or working diagnosis of dementia is noted. The patient is able to be contacted directly.

---
[Retrieved Note #4 | distance=0.8699]
patient_num=H117326759 | visit_id=6432799125 | note_csn_id=3768256165
note_type=Progress Notes | create_dts_shifted=2017-09-23 14:37:00

Medication orders were reviewed, and reconciliation was performed with the health record on 9/23/17. The dates of service were from 8/25/17 to 10/24/17. This progress note reflects ongoing monitoring, which is especially important given the patient's prior diagnosis of dementia.

---
[Retrieved Note #5 | distance=0.8872]
patient_num=H115124574 | visit_id=6395210747 | note_csn_id=3382253493
note_type=Progress Notes | create_dts_shifted=2017-12-04 14:24:00

The patient was admitted to the inpatient unit on 12/4/2017 due to shortness of breath and congestive heart failure. Communication with the care management team was established on 12/4/2017. There is a prior diagnosis of dementia noted, for which clinical confidence is moderate. The patient remains hospitalized and is able to be contacted.


### 📋 LLM Output: Dementia Phenotype from Retrieved Notes

Here's a structured answer based on the provided notes, following your instructions:

*   **Patient H116882795**
    *   Dementia Phenotype: Yes
    *   Evidence:
        *   "prior diagnosis of dementia" (Note #1)
        *   "highly established diagnosis of dementia" (Note #2)
    *   Rationale: The notes explicitly state a prior diagnosis of dementia for this patient in multiple instances.
    *   Confidence: High

*   **Patient H115124574**
    *   Dementia Phenotype: Yes
    *   Evidence:
        *   "prior history or working diagnosis of dementia" (Note #3)
        *   "prior diagnosis of dementia noted, for which clinical confidence is moderate" (Note #5)
    *   Rationale: This patient has a documented history or working diagnosis of dementia, with moderate clinical confidence.
    *   Confidence: Medium

*   **Patient H117326759**
    *   Dementia Phenotype: Yes
    *   Evidence:
        *   "prior diagnosis of dementia" (Note #4)
    *   Rationale: This patient has a documented prior diagnosis of dementia.
    *   Confidence: High